# Neural Prefetcher Zoo v2: 5 Models on a REAL ChampSim Trace

**Pipeline (v2 -- real data, no synthetic crutch):**
1. On the lab machine, run `bash scripts/dump_trace.sh` -- this calls ChampSim with the `trace_dumper` prefetcher and produces `access_trace.<TRACE>.csv`.
2. Upload that CSV to Colab below.
3. Train MLP / Perceptron / CNN / LSTM / Transformer on the same `(PC hash + 4-delta history) -> next page offset` task.
4. Export `prefetch_list.txt` for the best model.
5. On the lab machine, run `bash scripts/run_nn_replay.sh` -- this calls ChampSim with the `list_replayer` prefetcher to get REAL IPC numbers.

In [ ]:
import os, time, json, random
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print('device:', DEVICE)

## 1. Load the ChampSim CSV

Upload `access_trace.<TRACE>.csv` (produced by `dump_trace.sh`) using the Colab file panel, or set `CSV_PATH`.

In [ ]:
CSV_PATH = 'access_trace.605.mcf_s-994B.csv'   # change to your filename
MAX_ROWS = 1_000_000                            # cap for Colab memory

df = pd.read_csv(CSV_PATH, nrows=MAX_ROWS)
df['addr'] = df['addr_hex'].apply(lambda s: int(s, 16)).astype('int64')
df['pc']   = df['pc_hex'].apply(lambda s: int(s, 16)).astype('int64')
print(df.head())
print('rows:', len(df), '| unique PCs:', df.pc.nunique(), '| unique pages:', (df.addr // (1<<12)).nunique())

## 2. Featurize: PC hash + last 4 deltas (per-PC), label = next cache-line offset within current page

This is the exact feature set used by UBC MLP 2021, Voyager 2021, and DART 2024.

In [ ]:
HIST = 4
NUM_PC_HASH = 4096
DELTA_VOCAB = 257
NUM_CLASSES = 64        # 64 cache lines per 4 KiB page
PAGE_BITS = 12; LINE_BITS = 6

def qd(d):
    n = d >> LINE_BITS
    if n == 0: return 0
    if 0 < n <= 127:   return n
    if -126 <= n < 0:  return 128 + (-n)
    return 255

addrs = df['addr'].values
pcs   = df['pc'].values

last_addrs = {}                        # PC -> list of last addrs
Xpc = np.zeros(len(df), dtype=np.int64)
Xd  = np.zeros((len(df), HIST), dtype=np.int64)
Y   = np.zeros(len(df), dtype=np.int64)
Idx = np.zeros(len(df), dtype=np.int64)      # global access index, NOT the row
Page = np.zeros(len(df), dtype=np.int64)
keep = np.zeros(len(df), dtype=bool)

for i in range(len(df) - 1):
    pc = int(pcs[i]); addr = int(addrs[i]); addr_next = int(addrs[i+1])
    prev = last_addrs.get(pc, [])
    if len(prev) >= 1:
        d_hist = [qd(prev[-(k+1)] - prev[-(k+2)]) if len(prev) > k+1 else 0
                  for k in range(HIST)]
        Xpc[i]  = pc & (NUM_PC_HASH - 1)
        Xd[i]   = d_hist
        # Label = the next line offset (within the same 4 KiB page, modulo).
        Y[i]    = (addr_next >> LINE_BITS) & (NUM_CLASSES - 1)
        Idx[i]  = i                # global access counter the replayer uses
        Page[i] = addr_next >> PAGE_BITS
        keep[i] = True
    prev.append(addr); prev = prev[-8:]; last_addrs[pc] = prev

Xpc, Xd, Y, Idx, Page = Xpc[keep], Xd[keep], Y[keep], Idx[keep], Page[keep]
print('examples:', len(Y), '| label hist:',
      np.bincount(Y, minlength=NUM_CLASSES)[:10])

In [ ]:
class PFDS(Dataset):
    def __init__(self, pc, d, y): self.pc=torch.from_numpy(pc); self.d=torch.from_numpy(d); self.y=torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.pc[i], self.d[i], self.y[i]

ds = PFDS(Xpc, Xd, Y)
n_tr = int(0.8*len(ds))
tr, va = random_split(ds, [n_tr, len(ds)-n_tr], generator=torch.Generator().manual_seed(0))
BATCH = 512
tr_ld = DataLoader(tr, batch_size=BATCH, shuffle=True, drop_last=True)
va_ld = DataLoader(va, batch_size=BATCH, shuffle=False)
print('train/val:', len(tr), len(va))

## 3. Five models -- same I/O, comparable parameter counts

In [ ]:
EMB_PC, EMB_D = 32, 16

class Perceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.epc=nn.Embedding(NUM_PC_HASH,EMB_PC); self.ed=nn.Embedding(DELTA_VOCAB,EMB_D)
        self.fc=nn.Linear(EMB_PC+HIST*EMB_D, NUM_CLASSES)
    def forward(self,pc,d):
        return self.fc(torch.cat([self.epc(pc), self.ed(d).flatten(1)], 1))

class MLP(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.epc=nn.Embedding(NUM_PC_HASH,EMB_PC); self.ed=nn.Embedding(DELTA_VOCAB,EMB_D)
        D=EMB_PC+HIST*EMB_D
        self.net=nn.Sequential(nn.Linear(D,h),nn.ReLU(),nn.Linear(h,h),nn.ReLU(),nn.Linear(h,NUM_CLASSES))
    def forward(self,pc,d):
        return self.net(torch.cat([self.epc(pc), self.ed(d).flatten(1)], 1))

class CNN1D(nn.Module):
    def __init__(self, ch=32):
        super().__init__()
        self.epc=nn.Embedding(NUM_PC_HASH,EMB_PC); self.ed=nn.Embedding(DELTA_VOCAB,EMB_D)
        self.conv=nn.Sequential(nn.Conv1d(EMB_D,ch,2,padding=1),nn.ReLU(),
                                nn.Conv1d(ch,ch,2,padding=1),nn.ReLU(),
                                nn.AdaptiveAvgPool1d(1))
        self.fc=nn.Linear(ch+EMB_PC, NUM_CLASSES)
    def forward(self,pc,d):
        c=self.conv(self.ed(d).transpose(1,2)).squeeze(-1)
        return self.fc(torch.cat([self.epc(pc), c], 1))

class LSTMNet(nn.Module):
    def __init__(self, h=64):
        super().__init__()
        self.epc=nn.Embedding(NUM_PC_HASH,EMB_PC); self.ed=nn.Embedding(DELTA_VOCAB,EMB_D)
        self.lstm=nn.LSTM(EMB_D,h,batch_first=True); self.fc=nn.Linear(h+EMB_PC, NUM_CLASSES)
    def forward(self,pc,d):
        _,(hn,_)=self.lstm(self.ed(d))
        return self.fc(torch.cat([self.epc(pc), hn[0]], 1))

class TinyTransformer(nn.Module):
    def __init__(self, dm=32, nh=4, L=2):
        super().__init__()
        self.epc=nn.Embedding(NUM_PC_HASH,EMB_PC); self.ed=nn.Embedding(DELTA_VOCAB,dm)
        self.pos=nn.Parameter(torch.randn(HIST,dm)*0.02)
        enc=nn.TransformerEncoderLayer(d_model=dm,nhead=nh,dim_feedforward=64,batch_first=True,dropout=0.0)
        self.enc=nn.TransformerEncoder(enc,num_layers=L); self.fc=nn.Linear(dm+EMB_PC, NUM_CLASSES)
    def forward(self,pc,d):
        z=self.enc(self.ed(d)+self.pos).mean(1)
        return self.fc(torch.cat([self.epc(pc), z], 1))

MODELS = {'Perceptron':Perceptron, 'MLP':MLP, 'CNN':CNN1D, 'LSTM':LSTMNet, 'Transformer':TinyTransformer}
for n,cls in MODELS.items():
    m=cls(); print(f'{n:12s}  params = {sum(p.numel() for p in m.parameters()):>10,d}')

## 4. Train all 5

In [ ]:
EPOCHS = 5; LR = 2e-3

def eval_acc(m, ld):
    m.eval(); ok=tot=0
    with torch.no_grad():
        for pc,d,y in ld:
            pc,d,y=pc.to(DEVICE),d.to(DEVICE),y.to(DEVICE)
            ok += (m(pc,d).argmax(1)==y).sum().item(); tot += y.numel()
    return ok/tot

results = {}
for name, cls in MODELS.items():
    m = cls().to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=LR)
    t0=time.time()
    for ep in range(EPOCHS):
        m.train()
        for pc,d,y in tr_ld:
            pc,d,y=pc.to(DEVICE),d.to(DEVICE),y.to(DEVICE)
            loss = F.cross_entropy(m(pc,d), y)
            opt.zero_grad(); loss.backward(); opt.step()
        acc = eval_acc(m, va_ld)
        print(f'  [{name}] epoch {ep+1}/{EPOCHS}  val_acc={acc:.4f}')
    results[name] = {'model':m, 'va_acc':acc, 'train_s':time.time()-t0,
                     'params':sum(p.numel() for p in m.parameters())}
    print(f'  -> {name}: acc={acc:.4f}  params={results[name]["params"]:,}  t={results[name]["train_s"]:.1f}s')

print('\n' + '='*64)
print(f'{"Model":12s} {"Params":>10s}  {"Val Acc":>8s}  {"Train (s)":>10s}')
for n,r in results.items():
    print(f'{n:12s} {r["params"]:>10,d}  {r["va_acc"]:>8.4f}  {r["train_s"]:>10.1f}')

## 5. CPU-inference latency micro-benchmark

In [ ]:
def bench(m, n=2000):
    m.eval().to('cpu'); pc=torch.zeros(1,dtype=torch.long); d=torch.zeros(1,HIST,dtype=torch.long)
    with torch.no_grad():
        for _ in range(20): m(pc,d)
        t=time.perf_counter()
        for _ in range(n): m(pc,d)
        return (time.perf_counter()-t)/n*1e6

for n,r in results.items():
    r['inf_us'] = bench(r['model'])
    print(f'{n:12s}  CPU inf = {r["inf_us"]:6.1f} us')

## 6. Export prefetch_list.txt for the best model

Format the `list_replayer` C++ module expects:
```
access_idx  0xhex_prefetch_addr
```
`access_idx` is the same counter the dumper used. Predicted line offset is combined with the page address of the *next* access to produce a concrete prefetch target.

In [ ]:
best_name = max(results, key=lambda k: results[k]['va_acc'])
print('Best:', best_name)
best = results[best_name]['model'].to(DEVICE).eval()

OUT_PATH = 'prefetch_list.txt'
n_lines = 0
BS = 2048
with open(OUT_PATH, 'w') as fh, torch.no_grad():
    for i in range(0, len(Xpc), BS):
        pc = torch.from_numpy(Xpc[i:i+BS]).to(DEVICE)
        d  = torch.from_numpy(Xd[i:i+BS]).to(DEVICE)
        preds = best(pc, d).argmax(1).cpu().numpy()
        idxs  = Idx[i:i+BS]
        pages = Page[i:i+BS]
        for j in range(len(preds)):
            pf_addr = (int(pages[j]) << PAGE_BITS) | (int(preds[j]) << LINE_BITS)
            fh.write(f'{int(idxs[j])} 0x{pf_addr:x}\n')
            n_lines += 1
print('wrote', OUT_PATH, 'lines:', n_lines)

## 7. Chart + JSON summary

In [ ]:
import matplotlib.pyplot as plt
names=list(results); accs=[results[n]['va_acc'] for n in names]
lats=[results[n]['inf_us'] for n in names]
fig, a1 = plt.subplots(figsize=(8,4))
x = np.arange(len(names))
a1.bar(x-0.2, accs, 0.4, color='steelblue', label='val acc')
a1.set_ylabel('val acc'); a1.set_xticks(x); a1.set_xticklabels(names)
for i,a in enumerate(accs): a1.text(i-0.2, a+0.005, f'{a:.2f}', ha='center', fontsize=8)
a2 = a1.twinx(); a2.plot(x, lats, color='darkorange', marker='o', label='CPU us')
a2.set_ylabel('CPU inf us'); a2.set_yscale('log')
a1.set_title('NN Prefetcher Family (real ChampSim trace)')
fig.tight_layout(); fig.savefig('nn_family_comparison.png', dpi=140); plt.show()

summary = {n:{k:v for k,v in r.items() if k!='model'} for n,r in results.items()}
summary['_best'] = best_name
with open('nn_family_summary.json','w') as fh: json.dump(summary, fh, indent=2)
print('saved nn_family_comparison.png + nn_family_summary.json + prefetch_list.txt')
print('Now download prefetch_list.txt to the lab machine and run scripts/run_nn_replay.sh')